# Transcription Result

> Standardized result DTO for the transcription task — the data noun tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

**Canonical home (execution stage 8 — Option C / PILLAR 1c):** `TranscriptionResult` lives here in `cjm-capability-primitives` so a pure-compute transcription tool capability depends only on this data noun, never on the adapter machinery (`TaskAdapter`, the tool protocol, persistence). It relocated here from `cjm-transcription-adapter-interface.core` (which now re-exports it class-identically as a REMOVE-AFTER-OVERHAUL shim) and, before that, from `cjm-transcription-plugin-system`. Field/key + wire-kind parity (`"transcription.result"`) is load-bearing: existing capability databases and fused-era plugins must keep working against this class.

In [ ]:
#| default_exp transcription

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_plugin_system.core.wire import wire_type

In [ ]:
#| export
@wire_type("transcription.result")
@dataclass
class TranscriptionResult:
    """Standardized output for all transcription plugins."""
    text: str                                        # The transcribed text
    confidence: Optional[float] = None               # Overall confidence (0.0 to 1.0)
    segments: Optional[List[Dict[str, Any]]] = None  # Timestamped segments
    metadata: Dict[str, Any] = field(default_factory=dict)  # Additional metadata

In [ ]:
# Test TranscriptionResult
result = TranscriptionResult(
    text="Hello world",
    confidence=0.95,
    segments=[
        {"start": 0.0, "end": 0.5, "text": "Hello"},
        {"start": 0.5, "end": 1.0, "text": "world"}
    ],
    metadata={"model": "whisper-large-v3", "language": "en"}
)

print(f"Text: {result.text}")
print(f"Confidence: {result.confidence}")
print(f"Segments: {result.segments}")
print(f"Metadata: {result.metadata}")

Text: Hello world
Confidence: 0.95
Segments: [{'start': 0.0, 'end': 0.5, 'text': 'Hello'}, {'start': 0.5, 'end': 1.0, 'text': 'world'}]
Metadata: {'model': 'whisper-large-v3', 'language': 'en'}


In [ ]:
# Test minimal result (only text required)
minimal = TranscriptionResult(text="Just the text")
print(f"Minimal result: {minimal}")

Minimal result: TranscriptionResult(text='Just the text', confidence=None, segments=None, metadata={})


In [ ]:
# Wire-format executable test: the result round-trips TYPED through the
# simulated worker boundary (encode -> json -> decode), and the flat
# from_dict default attached by @wire_type reconstructs it.
import json as _json
from cjm_plugin_system.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

res = TranscriptionResult(
    text="It's one small step for man,",
    confidence=None,
    segments=[{"start": 0.0, "end": 2.5, "text": "It's one small step for man,"}],
    metadata={"model": "whisper-base", "language": "en"},
)
envelope = wire_encode(res)
assert envelope[WIRE_KIND_KEY] == "transcription.result"
back = wire_decode(_json.loads(_json.dumps(envelope)))
assert isinstance(back, TranscriptionResult) and back == res
print("transcription.result wire round-trip OK")

transcription.result wire round-trip OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()